In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms.functional as VF
from PIL import Image
import time

device = 'cuda'
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.8.0+cu128


In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [3]:
train_dataset = datasets.MNIST(
    root="./DATA",
    train=True,
    download=True,
    transform=transform)

test_dataset = datasets.MNIST(
    root="./DATA",
    train=False,
    download=True,
    transform=transform)    

In [4]:
# tensorboard --logdir runs --bind_all --load_fast=false --samples_per_plugin images=100
#writer = SummaryWriter('runs/exp')
#for j in range(10):
#    for i in range(0, 90, 5):
#        image, label = train_dataset[j]
#        image = VF.rotate(image, i)
#        fig = plt.figure(figsize=(16, 4))
#        plt.imshow(image, cmap="gray")
#        plt.title(f"Label: {label}")
#        writer.add_figure(f'Rotate_{j}', fig, global_step=i)
#        plt.close(fig)
#writer.close()        

In [5]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)        
        return x

In [6]:
def test_model(model, dataloader, criterion):
    model.eval()
    running_loss_test = 0.0
    n_obs_test = 0
    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        running_loss_test += loss.item() * len(data)
        n_obs_test += len(data)
    epoch_loss_test = running_loss_test / n_obs_test
    return epoch_loss_test

In [15]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False)

model = SimpleNN().to(device)
optimizer = optim.Adam(
    params=model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)
criterion = nn.CrossEntropyLoss()

writer = SummaryWriter(f'runs/model_{int(time.time())}')
global_step = 0
for epoch in range(10):
    running_loss = 0.0
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        running_loss += loss.item()

        # Log Params Batch
        total_param_norm = 0.0
        total_param_count = 0.0
        total_grad_norm = 0.0
        for name, param in model.named_parameters():
            if param.requires_grad:
                if "bias" in name:
                    short_name = name[:-5]
                    group_name = "Bias"
                else:
                    short_name = name[:-7]
                    group_name = "Weight"
                
                writer.add_scalar(f"{group_name}/{short_name}_norm_scaled", param.data.norm(2) / (param.data.numel() ** 0.5), global_step) 
                writer.add_scalar(f"Gradient_{group_name}/{short_name}_norm", param.grad.data.norm(2), global_step) 
                writer.add_scalar(f"Update_ratio_{group_name}/{short_name}", param.grad.data.norm(2) / param.data.norm(2), global_step)

                total_param_norm += param.data.norm(2) ** 2
                total_param_count += param.data.numel()
                total_grad_norm += param.grad.data.norm(2) ** 2
        writer.add_scalar(f"Total_norm/Param_scaled", (total_param_norm / total_param_count) ** 0.5, global_step) 
        writer.add_scalar(f"Total_norm/Gradient", total_grad_norm ** 0.5, global_step)
        writer.add_scalar("Loss/train/batch", loss.item(), global_step)
        
        global_step += 1
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step() 

    # Log Params Epoch
    for name, param in model.named_parameters():
        if param.requires_grad:
            if "bias" in name:
                short_name = name[:-5]
                group_name = "Bias"
            else:
                short_name = name[:-7]
                group_name = "Weight"
        writer.add_histogram(f"{group_name}/{short_name}_epoch", param.data.cpu(), epoch)
        writer.add_histogram(f"Gradient_{group_name}/{short_name}_epoch", param.grad.data.cpu(), epoch)
    epoch_loss = running_loss / len(train_loader)
    #writer.add_scalar("Loss_epoch/train", epoch_loss, epoch)

    loss_test_epoch = test_model(model = model, dataloader = test_loader, criterion = criterion)
    #writer.add_scalar("Loss_epoch/test", loss_test_epoch, epoch)
    writer.add_scalars("Loss2/epoch", {"train": epoch_loss, "test": loss_test_epoch}, epoch)
    
writer.close()